# A priori vs. a posteriori: maps, false discoveries, and Wilks' FDR

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eabarnes1010/course_objective_analysis/blob/main/code/maps_of_random_relationships.ipynb)

*Elizabeth A. Barnes — BU Earth & Environment — Last updated: 2026-05-22*

## Learning objectives

- See why **global maps of grid-point significance can deceive**: even under a true null, ~5% of points will pop out at the 95% level — and spatial correlation makes them appear in clusters.
- Correlate a global geopotential height field against a totally random time series and observe these spurious "signals" first-hand.
- Apply **Wilks' (2016) False Discovery Rate** procedure to obtain a global p-value threshold that controls the expected fraction of false positives across the map.

---

## Motivation

Geophysical results are often presented as maps with stippled "significant" grid points. Two things make these maps treacherous:

1. **Multiple comparisons.** At a 95% confidence level, *5% of all grid points* will be flagged significant under the true null hypothesis.
2. **Spatial correlation.** Atmospheric fields are smooth — so those false-positive points tend to clump into plausible-looking patches.

To demonstrate, we correlate daily January 500 hPa geopotential heights against a completely random time series `Z`, and watch the map light up.


In [ ]:
try:
    import google.colab

    IN_COLAB = True
except ImportError:
    IN_COLAB = False
print(f"IN_COLAB = {IN_COLAB}")

In [ ]:
if IN_COLAB:
    # cartopy is preinstalled on modern Colab; if not, fall back to pip.
    try:
        import cartopy  # noqa: F401
    except ImportError:
        get_ipython().system("pip install cartopy")

import random

import cartopy as ct
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import scipy.io as sio
import scipy.stats as stats

mpl.rcParams["figure.dpi"] = 100

random.seed(30)
np.random.seed(seed=16)


## Load the data

A Matlab `.mat` file containing daily Northern Hemisphere 500 hPa geopotential heights on a lat/lon grid.


In [ ]:
if IN_COLAB:
    get_ipython().system("pip install wget")
    import wget

    filename = wget.download(
        "https://scc1.bu.edu/~eabarnes/class/z500_daily.mat"
    )
else:
    filename = "../data/z500_daily.mat"

DATA = sio.loadmat(filename)
Xall = np.array(DATA["X"])
LAT = np.array(DATA["LAT"])[0, :]
LONG = np.array(DATA["LONG"])[0, :]
TIME = np.array(DATA["TIME"])
print("data is loaded")


### Thin and reshape

Grab every other lat/lon to keep the computation snappy, close the longitude seam at 360°E, and restrict to December 1 days only.


In [ ]:
# Thin the grid.
Xall = Xall[:, ::2, ::2]
LAT = LAT[::2]
LONG = LONG[::2]

# Close the longitude seam at 360E.
Xall = np.insert(Xall, 0, values=Xall[:, :, 0], axis=2)
LONG = np.insert(LONG, 0, values=0.0)
Xall = np.insert(Xall, -1, values=Xall[:, :, -1], axis=2)
LONG = np.insert(LONG, -1, values=LONG[-1])
LONG[-1] = 360
del DATA

# Restrict to December 1.
itime = np.logical_and(TIME[:, 2] == 12, TIME[:, 3] == 1)
X = Xall[itime, :, :]

## Build the random time series `Z`

`Z` has no knowledge of the atmosphere — it's pure Gaussian noise (with the same number of time steps as `X`). Toggle `randomSeries` to compare against an actual atmospheric time series from a single grid point.


In [ ]:
randomSeries = False  # True = pure noise; False = real Z500 time series at one point

if randomSeries:
    Z = np.random.normal(size=X.shape[0])
else:
    print(f"latitude = {LAT[5]}; longitude = {LONG[0]}")
    Z = X[:, 5, 0]
Z = Z - np.mean(Z)

plt.figure()
plt.plot(Z)
plt.title(f"randomSeries = {randomSeries}")
plt.show()

## Correlate every grid point against `Z`

Store both the correlation `C[lat, lon]` and the local p-value `P[lat, lon]`.


In [ ]:
P = np.empty((X.shape[1], X.shape[2]))
C = np.empty((X.shape[1], X.shape[2]))

for ilat in range(X.shape[1]):
    for ilon in range(X.shape[2]):
        corr_val, p = stats.pearsonr(Z, X[:, ilat, ilon])
        C[ilat, ilon] = corr_val
        P[ilat, ilon] = p

## Plot the map

Shade by correlation; stipple points where `p < 0.05`.


In [ ]:
data_crs = ct.crs.PlateCarree()

plt.figure(figsize=(16 / 1.2, 8 / 1.2))
ax = plt.subplot(projection=ct.crs.PlateCarree())
ax.set_global()
ax.coastlines(linewidth=0.75)
image = ax.pcolor(LONG, LAT, C, transform=data_crs, cmap="RdBu_r")
cb = plt.colorbar(image, shrink=0.5, orientation="horizontal", pad=0.05)
cb.set_label("correlation", fontsize=16)
maxval = np.max(C)
image.set_clim(-maxval, maxval)

for ilat, vallat in enumerate(LAT):
    for ilon, vallon in enumerate(LONG):
        if P[ilat, ilon] < 0.05:
            ax.plot(
                vallon, vallat, "o", markersize=3, color="fuchsia", transform=data_crs
            )

plt.title("Z500 correlations", fontsize=20)
plt.show()

The stippling clusters into patches that *look* meaningful, even though `Z` is random (when `randomSeries = True`) or only weakly related to most of the field. This is the heart of the multiple-comparisons problem in earth-science maps.

The fix is to think physically (is there a mechanism that would explain this?) **and** to apply a procedure that controls global error.

## Wilks' false discovery rate

[Wilks (2016, BAMS)](http://journals.ametsoc.org/doi/abs/10.1175/BAMS-D-15-00267.1) gives a simple way to control the **false discovery rate (FDR)** — the expected fraction of rejected local null hypotheses that are actually true. Procedure:

1. Sort the local p-values in increasing order: `p_(1) ≤ p_(2) ≤ ... ≤ p_(N)`.
2. Pick a target FDR level `alpha_FDR`.
3. Find the largest `i` such that `p_(i) ≤ (i / N) * alpha_FDR`.
4. That `p_(i)` is the **global p-value threshold**. Only grid points with `p < p_(i)` are considered globally significant.

Visually it's the intersection of the sorted-p curve with the line `y = (i/N) * alpha_FDR`.


In [ ]:
alpha = 0.2

Pvals = np.sort(P.flatten())
x = np.arange(1, len(Pvals) + 1, dtype=float)
y = (x / len(x)) * alpha

# Find the FDR cutoff: largest index where Pvals stays under the line.
d = Pvals - y
above_idx = np.where(d > 0)[0]
k = above_idx[0] if len(above_idx) > 0 else 0
if k == 0:
    fdrPcutoff = 0.0
else:
    fdrPcutoff = Pvals[k - 1]

if randomSeries:
    xlimVal, ylimVal = 6200, 0.4
else:
    xlimVal, ylimVal = 200, 0.08

plt.figure(figsize=(10, 5))
plt.plot(x, Pvals, ".", color="dimgray", markersize=5, label="actual p-values")
plt.plot(x, y, "--", color="cornflowerblue", label="FDR criterion")
if k > 0:
    plt.plot([k, k], [0, 1], "--", color="gray")
    plt.annotate(
        f"$p_{{crit}}$ = {fdrPcutoff:.3f}",
        xy=(k, Pvals[k - 1]),
        xytext=(xlimVal / 2, ylimVal / 2.0),
        arrowprops=dict(
            facecolor="red", shrink=0.08, width=0.5, headlength=10, headwidth=7
        ),
        horizontalalignment="left",
        color="red",
    )
plt.xlim(0, xlimVal)
plt.ylim(0, ylimVal)
plt.xlabel("index")
plt.ylabel("p-value")
plt.title("Sorted p-values vs. FDR criterion")
plt.legend()
plt.show()

If the sorted p-values never cross below the FDR line, no point is globally significant.

### Map with the FDR cutoff overlaid

Magenta circles = local 95% significance. White X = grid point also passing the global FDR cutoff.


In [ ]:
plt.figure(figsize=(16 / 1.2, 8 / 1.2))
ax = plt.subplot(projection=ct.crs.PlateCarree())
ax.set_global()
ax.coastlines(linewidth=0.75)
image = ax.pcolor(LONG, LAT, C, transform=data_crs, cmap="RdBu_r")
cb = plt.colorbar(image, shrink=0.5, orientation="horizontal", pad=0.05)
cb.set_label("correlation", fontsize=16)
maxval = np.max(C)
image.set_clim(-maxval, maxval)

for ilat, vallat in enumerate(LAT):
    for ilon, vallon in enumerate(LONG):
        if P[ilat, ilon] < 0.05:
            ax.plot(
                vallon, vallat, "o", markersize=3, color="fuchsia", transform=data_crs
            )
        if P[ilat, ilon] < fdrPcutoff:
            ax.plot(
                vallon, vallat, "x", markersize=3, color="white", transform=data_crs
            )

plt.title("Z500 correlations with Wilks FDR overlay", fontsize=20)
plt.show()

## Wrap-up

- "5% of my map is significant" is the **expected** result under the null at the 95% level — it does not mean anything is real.
- Spatial correlation makes false positives cluster, which can be very misleading.
- Wilks' FDR procedure converts the row of local p-values into a single, principled global threshold.
- Always pair statistical thresholds with a physical argument before claiming a signal.
